In [1]:
# Install if needed
!pip install tensorflow

# Imports
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
print("Loading movie review dataset...")

VOCAB_LIMIT = 12000
SEQ_LEN = 180

(train_X, train_y), (test_X, test_y) = imdb.load_data(num_words=VOCAB_LIMIT)

# Padding sequences
train_X = pad_sequences(train_X, maxlen=SEQ_LEN, padding='post')
test_X = pad_sequences(test_X, maxlen=SEQ_LEN, padding='post')

print("Train size:", train_X.shape)

Loading movie review dataset...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train size: (25000, 180)


In [3]:
X_train_t = torch.LongTensor(train_X)
y_train_t = torch.FloatTensor(train_y)

X_test_t = torch.LongTensor(test_X)
y_test_t = torch.FloatTensor(test_y)

BATCH_SIZE = 128

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=BATCH_SIZE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [4]:
class BiDirectionalLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()

        self.embedding_layer = nn.Embedding(vocab_size, embed_dim)

        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.output_layer = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding_layer(x)

        _, (hidden, _) = self.bilstm(x)

        # Extract forward & backward hidden states
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        combined = torch.cat((forward_hidden, backward_hidden), dim=1)

        out = self.output_layer(combined)
        return out.squeeze()

In [5]:
model = BiDirectionalLSTM(VOCAB_LIMIT).to(device)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 6

In [6]:
print("Training started...\n")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    correct = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds > 0.5).float()
        correct += (preds == labels).sum().item()

    avg_loss = epoch_loss / len(train_loader)
    acc = correct / len(train_loader.dataset)

    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Accuracy: {acc*100:.2f}%")

Training started...

Epoch 1 | Loss: 0.6347 | Accuracy: 61.93%
Epoch 2 | Loss: 0.5904 | Accuracy: 67.76%
Epoch 3 | Loss: 0.5143 | Accuracy: 75.46%
Epoch 4 | Loss: 0.4269 | Accuracy: 81.20%
Epoch 5 | Loss: 0.3565 | Accuracy: 84.95%
Epoch 6 | Loss: 0.2620 | Accuracy: 89.70%


In [7]:
model.eval()

total_loss = 0
correct = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = loss_fn(outputs, labels)

        total_loss += loss.item()

        preds = torch.sigmoid(outputs)
        preds = (preds > 0.5).float()
        correct += (preds == labels).sum().item()

final_loss = total_loss / len(test_loader)
final_acc = correct / len(test_loader.dataset)

print("\nFinal Results:")
print(f"Test Loss: {final_loss:.4f}")
print(f"Test Accuracy: {final_acc*100:.2f}%")


Final Results:
Test Loss: 0.3434
Test Accuracy: 85.59%
